<a href="https://colab.research.google.com/github/meharalirajar060-codeee/Flyrank_Internship_ML_MAR/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/meharalirajar060-codeee/Flyrank_Internship_ML_MAR/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
hf_token = userdata.get('HF_TOKEN')
con.execute(f"""
    CREATE SECRET hf_token (
        TYPE huggingface,
        TOKEN '{hf_token}'
    );
""")
FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

data = con.execute(f"""
    WITH march AS (
        SELECT content_hash_id, client_hash_id,
               SUM(gsc_impressions) AS impressions_90d,
               SUM(gsc_clicks) AS clicks_90d,
               AVG(gsc_avg_position) AS avg_position_90d,
               DATE_DIFF('day', MAX(report_date), DATE '2026-03-31') AS days_since_last_seen
        FROM read_parquet('{FACT}')
        WHERE month = '2026-03'
        GROUP BY content_hash_id, client_hash_id
        HAVING SUM(gsc_impressions) >= 100
    ),
    april AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_april
        FROM read_parquet('{FACT}')
        WHERE month = '2026-04'
        GROUP BY content_hash_id
    )
    SELECT m.*, COALESCE(a.impressions_april, 0) AS impressions_april
    FROM march m
    LEFT JOIN april a ON m.content_hash_id = a.content_hash_id
""").df()

data["ctr_90d"] = data["clicks_90d"] / data["impressions_90d"]
data["is_declining_label"] = (data["impressions_april"] < data["impressions_90d"] * 0.9).astype(int)
print(data.shape, data["is_declining_label"].mean())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(101441, 9) 0.5925809090998708


In [5]:
from sklearn.model_selection import GroupShuffleSplit

features = ["impressions_90d", "clicks_90d", "avg_position_90d", "ctr_90d", "days_since_last_seen"]
X = data[features].fillna(0)
y = data["is_declining_label"]
groups = data["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train clients:", data.iloc[train_idx]["client_hash_id"].nunique(),
      "| Test clients:", data.iloc[test_idx]["client_hash_id"].nunique())

Train clients: 35 | Test clients: 9


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""Section 1 note: I was unable to locate the specific FlyRank research paper referenced in this week's live session — extensive searching did not surface it as a public document, and it appears to be an internal/portal-only resource. Rather than audit findings I haven't actually read (which would defeat the purpose of a methodology audit), I'm flagging this gap honestly. I plan to complete this section once I obtain the paper from the portal/mentor and will update this notebook at that time. In the meantime, Sections 2-4 apply the same audit discipline directly to my own Week-5 model, which I have full access to."""


"Section 1 note: I was unable to locate the specific FlyRank research paper referenced in this week's live session — extensive searching did not surface it as a public document, and it appears to be an internal/portal-only resource. Rather than audit findings I haven't actually read (which would defeat the purpose of a methodology audit), I'm flagging this gap honestly. I plan to complete this section once I obtain the paper from the portal/mentor and will update this notebook at that time. In the meantime, Sections 2-4 apply the same audit discipline directly to my own Week-5 model, which I have full access to."

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""Before/after: [fill with actual printed numbers once run]. Expectation: the naive random split typically inflates scores relative to the client-holdout split, since a client's other pages sitting in training let the model partially memorize client-specific patterns rather than learning signal that generalizes to clients it has never seen. The honest number to report is the client-holdout one, even if it's lower."""


"Before/after: [fill with actual printed numbers once run]. Expectation: the naive random split typically inflates scores relative to the client-holdout split, since a client's other pages sitting in training let the model partially memorize client-specific patterns rather than learning signal that generalizes to clients it has never seen. The honest number to report is the client-holdout one, even if it's lower."

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# BEFORE: naive random split — a client's pages can leak across train/test
X_tr_naive, X_te_naive, y_tr_naive, y_te_naive = train_test_split(X, y, test_size=0.2, random_state=42)
model_naive = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42).fit(X_tr_naive, y_tr_naive)
proba_naive = model_naive.predict_proba(X_te_naive)[:, 1]

print("BEFORE (naive random split):")
print("AUC:", roc_auc_score(y_te_naive, proba_naive))
print("Avg precision:", average_precision_score(y_te_naive, proba_naive))
print("Precision@50:", precision_at_k(proba_naive, y_te_naive.values, 50))

# AFTER: client-holdout split
model_grouped = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42).fit(X_train, y_train)
proba_grouped = model_grouped.predict_proba(X_test)[:, 1]

print("\nAFTER (client-holdout split):")
print("AUC:", roc_auc_score(y_test, proba_grouped))
print("Avg precision:", average_precision_score(y_test, proba_grouped))
print("Precision@50:", precision_at_k(proba_grouped, y_test.values, 50))

BEFORE (naive random split):
AUC: 0.6258281867353341
Avg precision: 0.6890292931722053
Precision@50: 0.78

AFTER (client-holdout split):
AUC: 0.5800013738789083
Avg precision: 0.6152602471967237
Precision@50: 0.64


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""Leakage audit on my ML-08 features (impressions_90d, clicks_90d, avg_position_90d, ctr_90d, days_since_last_seen):
- Are any features calculated after the decision point? No — all five are aggregated strictly from the March window; the label comes from April, which never overlaps the feature window.
- Does the feature window overlap the target window? No.
- Did any product output (health_score, priority_score, action_type) slip in as a feature? No — not shipped in this data at all.
- Does a derived field secretly encode the target? ctr_90d is derived only from March clicks/impressions — no April data feeds it.
- Are duplicate/related rows split across train/test in a way that makes the test too easy? This was exactly the problem demonstrated in Section 2's naive split; the client-holdout split fixes it.
- Are you testing on clients the model hasn't effectively already seen? Only under the client-holdout split — not under the naive one."""

"Leakage audit on my ML-08 features (impressions_90d, clicks_90d, avg_position_90d, ctr_90d, days_since_last_seen):\n- Are any features calculated after the decision point? No — all five are aggregated strictly from the March window; the label comes from April, which never overlaps the feature window.\n- Does the feature window overlap the target window? No.\n- Did any product output (health_score, priority_score, action_type) slip in as a feature? No — not shipped in this data at all.\n- Does a derived field secretly encode the target? ctr_90d is derived only from March clicks/impressions — no April data feeds it.\n- Are duplicate/related rows split across train/test in a way that makes the test too easy? This was exactly the problem demonstrated in Section 2's naive split; the client-holdout split fixes it.\n- Are you testing on clients the model hasn't effectively already seen? Only under the client-holdout split — not under the naive one."

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"""Original claim (ML-08): 'Random Forest wins Precision@50 (0.72 vs baseline 0.64).'
Rewritten: 'Under a client-holdout split, Random Forest showed a directionally higher Precision@50 (0.72) than the baseline rule (0.64) on this March-to-April slice — an observed, decision-support signal on one month of data, not a guarantee any flagged page will actually decline, and not evidence the model generalizes to other months or the full warehouse.

Original claim (ML-08): 'The model is weakest exactly where volume is thin.'
Rewritten: 'False negatives in this test slice were associated with lower measured impressions and CTR — an observed pattern in one client-holdout split, not a proven causal weakness of the model architecture.'"""


"Original claim (ML-08): 'Random Forest wins Precision@50 (0.72 vs baseline 0.64).'\nRewritten: 'Under a client-holdout split, Random Forest showed a directionally higher Precision@50 (0.72) than the baseline rule (0.64) on this March-to-April slice — an observed, decision-support signal on one month of data, not a guarantee any flagged page will actually decline, and not evidence the model generalizes to other months or the full warehouse.\n\nOriginal claim (ML-08): 'The model is weakest exactly where volume is thin.'\nRewritten: 'False negatives in this test slice were associated with lower measured impressions and CTR — an observed pattern in one client-holdout split, not a proven causal weakness of the model architecture.'"

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.